In [ ]:
from stable_baselines3.ppo import PPO
import dill
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO

cm_to_inch = 1 / 2.54  # centimeters in inches

In [ ]:
data_path = r"..\experiments\split_input_rl\cons_100\res_rl_split_input_rl_seed_1.dat"
model_path = r"..\experiments\split_input_rl\models\split_input_rl_1.mdl"
red_model_path = r"..\experiments\measured_input_rl\models\measured_input_rl_1.mdl"

In [ ]:
with open(data_path, "rb") as ff:
    data = dill.load(ff)
model = PPO.load(model_path)

In [ ]:
# The load can go from -5 to +5
def generate_policy_vis(time, weather):
    t = [time]
    prev_weather = [weather] * 4
    other_obs = [0.5] * 8
    load_xx = np.linspace(-4, 4, 100)
    acts = np.zeros((100, 100))
    for ax1, loc_ld in enumerate(load_xx):
        for ax2, glob_ld in enumerate(load_xx):
            prev_local_load = [loc_ld] * 8
            prev_global_load = [glob_ld] * 8
            state = np.array(
                t + prev_local_load + prev_global_load + prev_weather + other_obs
            )
            acts[ax1][ax2] = model.predict(state, deterministic=True)[0].item()
    return acts

In [ ]:
# Generate the data
times = [0.1, 0.5, 0.9]
weathers = [-4.0, 0.0, 4.0]
plt_data = {k: {} for k in times}
for time in times:
    for weather in weathers:
        print(f"Generating time={time}, weather={weather}")
        plt_data[time][weather] = generate_policy_vis(time, weather)

In [ ]:
# Define labels for time and weather
time_labels = ["Morning", "Noon", "Evening"]
weather_labels = ["Cool", "Neutral", "Hot"]

# Create subplots
fig, axes = plt.subplots(3, 3, figsize=(19 * cm_to_inch, 12 * cm_to_inch))

# Set common color scale
vmin = np.min(
    [np.min(plt_data[time][weather]) for time in times for weather in weathers]
)
vmax = np.max(
    [np.max(plt_data[time][weather]) for time in times for weather in weathers]
)

# Loop over the times and weathers to plot each subplot
for i, time in enumerate(times):
    for j, weather in enumerate(weathers):
        # Get the corresponding axis
        ax = axes[j, i]

        # Plot the 2D array with a shared colorbar
        cax = ax.imshow(
            plt_data[time][weather],
            vmin=vmin,
            vmax=vmax,
            cmap="coolwarm",
            origin="lower",
        )
        # cax = ax.imshow(plt_data[time][weather]-plt_data[time][weathers[0]], vmin=vmin, vmax=vmax, cmap='jet', origin='lower')

        # Label axes as "Low" to "High" instead of numbers
        ax.set_xticks([0, 50, 99])
        ax.set_xticklabels(["", "", ""])
        ax.set_yticks([0, 50, 99])
        ax.set_yticklabels(["", "", ""])

        # Label the top center one to show what the axes mean
        if i == 1 and j == 1:
            ax.set_xlabel("Building Load", fontsize=8)
            ax.set_ylabel("Cluster Load", fontsize=8)
            # Label axes as "Low" to "High" instead of numbers
            ax.set_xticks([0, 50, 99])
            ax.set_xticklabels(["Low", "", "High"], fontsize=8)
            ax.set_yticks([0, 50, 99])
            ax.set_yticklabels(["Low", "", "High"], fontsize=8)

        # Only label the leftmost and bottom subplots for clarity
        if i == 0:
            ax.set_ylabel(weather_labels[j])  # Set weather label
        if j == 2:
            ax.set_xlabel(time_labels[i])  # Set time label

# Add common x and y labels for the inner axes
# fig.text(0.5, 0.04, 'Building Load', ha='center', va='center')
# fig.text(0.04, 0.5, 'Cluster Load', ha='center', va='center', rotation='vertical')
fig.supxlabel("Time of Day")
fig.supylabel("Weather")

# Add a common colorbar outside the subplots
fig.subplots_adjust(right=0.85)  # Make space for the colorbar
cbar_ax = fig.add_axes([0.88, 0.15, 0.02, 0.7])  # Create colorbar axis
cbar = fig.colorbar(cax, cax=cbar_ax)
cbar.set_label("Requested Setpoint")
cbar.set_ticks([vmin, 0, vmax])
cbar.set_ticklabels(["Lower", "Ideal", "Higher"])

# Adjust layout
fig.tight_layout(rect=[0, 0, 0.85, 1])  # Adjust to fit colorbar
# fig.savefig("export/policy_split_15d.png", dpi=400, bbox_inches="tight")

In [ ]:
red_model = PPO.load(red_model_path)

In [ ]:
# The load can go from -5 to +5
def generate_policy_vis_red(time, weather):
    t = [time]
    prev_weather = [weather] * 4
    load_xx = np.linspace(-4, 4, 100)
    acts = np.zeros((100, 100))
    for ax1, loc_ld in enumerate(load_xx):
        for ax2, glob_ld in enumerate(load_xx):
            prev_local_load = [loc_ld] * 8
            prev_global_load = [glob_ld] * 8
            state = np.array(t + prev_local_load + prev_global_load + prev_weather)
            acts[ax1][ax2] = red_model.predict(state, deterministic=True)[0].item()
    return acts

In [ ]:
# Generate the data
plt_data_red = {k: {} for k in times}
for time in times:
    for weather in weathers:
        print(f"Generating time={time}, weather={weather}")
        plt_data_red[time][weather] = generate_policy_vis_red(time, weather)

In [ ]:
# Define labels for time and weather
time_labels = ["Morning", "Noon", "Evening"]
weather_labels = ["Cool", "Neutral", "Hot"]

# Create subplots
fig, axes = plt.subplots(3, 3, figsize=(7, 5))

# Set common color scale
vmin = np.min(
    [np.min(plt_data_red[time][weather]) for time in times for weather in weathers]
)
vmax = np.max(
    [np.max(plt_data_red[time][weather]) for time in times for weather in weathers]
)

# Loop over the times and weathers to plot each subplot
for i, time in enumerate(times):
    for j, weather in enumerate(weathers):
        # Get the corresponding axis
        ax = axes[j, i]

        # Plot the 2D array with a shared colorbar
        cax = ax.imshow(
            plt_data_red[time][weather],
            vmin=vmin,
            vmax=vmax,
            cmap="coolwarm",
            origin="lower",
        )
        # cax = ax.imshow(plt_data[time][weather]-plt_data[time][weathers[0]], vmin=vmin, vmax=vmax, cmap='jet', origin='lower')

        # Label axes as "Low" to "High" instead of numbers
        ax.set_xticks([0, 50, 99])
        ax.set_xticklabels(["", "", ""])
        ax.set_yticks([0, 50, 99])
        ax.set_yticklabels(["", "", ""])

        # Label the top center one to show what the axes mean
        if i == 1 and j == 1:
            ax.set_xlabel("Building Load")
            ax.set_ylabel("Cluster Load")
            # Label axes as "Low" to "High" instead of numbers
            ax.set_xticks([0, 50, 99])
            ax.set_xticklabels(["Low", "", "High"])
            ax.set_yticks([0, 50, 99])
            ax.set_yticklabels(["Low", "", "High"])

        # Only label the leftmost and bottom subplots for clarity
        if i == 0:
            ax.set_ylabel(weather_labels[j])  # Set weather label
        if j == 2:
            ax.set_xlabel(time_labels[i])  # Set time label

# Add common x and y labels for the inner axes
# fig.text(0.5, 0.04, 'Building Load', ha='center', va='center')
# fig.text(0.04, 0.5, 'Cluster Load', ha='center', va='center', rotation='vertical')
fig.supxlabel("Time of Day")
fig.supylabel("Weather")

# Add a common colorbar outside the subplots
fig.subplots_adjust(right=0.85)  # Make space for the colorbar
cbar_ax = fig.add_axes([0.88, 0.15, 0.02, 0.7])  # Create colorbar axis
cbar = fig.colorbar(cax, cax=cbar_ax)
cbar.set_label("Requested Setpoint")
cbar.set_ticks([vmin, vmax])
cbar.set_ticklabels(["Lower", "Higher"])

# Adjust layout
fig.tight_layout(rect=[0, 0, 0.85, 1])  # Adjust to fit colorbar
# fig.savefig("export/policy_reduced_60d.png", dpi=400)

In [ ]:
get_transition_fraction = lambda arr, thresh: len(arr[abs(arr) < thresh]) / len(
    arr.ravel()
)

In [ ]:
# Example data (replace plt_data and plt_data_2 with your actual data)
plt_data_spl_sub = plt_data[0.5]
plt_data_red_sub = plt_data_red[0.5]

# Create figure and axes
fig, axes = plt.subplots(2, 2, figsize=(9 * cm_to_inch, 9 * cm_to_inch))
# fig.subplots_adjust(wspace=0.02, hspace=0.05)  # Adjust spacing between plots

# Plot data with coolwarm colormap
cmap = "coolwarm"
vmin, vmax = -1, 1  # Color scale limits

# Top-left: plt_data_spl_sub[4] ("Warm" and "Split-Input")
im = axes[0, 0].imshow(
    plt_data_spl_sub[-4], cmap=cmap, vmin=vmin, vmax=vmax, origin="lower"
)
axes[0, 0].text(
    55,
    85,
    f"{get_transition_fraction(plt_data_spl_sub[-4], 0.5) * 100.0:<4.2f}%",
    color="w",
    fontsize=11.0,
    weight="bold",
)

# Bottom-left: plt_data_spl_sub[-4] ("Cool" and "Split-Input")
axes[1, 0].imshow(plt_data_spl_sub[4], cmap=cmap, vmin=vmin, vmax=vmax, origin="lower")
axes[1, 0].text(
    55,
    85,
    f"{get_transition_fraction(plt_data_spl_sub[4], 0.5) * 100.0:<4.2f}%",
    color="w",
    fontsize=11.0,
    weight="bold",
)

# Top-right: plt_data_red_sub[4] ("Warm" and "Measured-Input")
axes[0, 1].imshow(plt_data_red_sub[-4], cmap=cmap, vmin=vmin, vmax=vmax, origin="lower")
axes[0, 1].text(
    47,
    85,
    f"{get_transition_fraction(plt_data_red_sub[-4], 0.5) * 100.0:<4.2f}%",
    color="w",
    fontsize=11.0,
    weight="bold",
)

# Bottom-right: plt_data_red_sub[-4] ("Cool" and "Measured-Input")
axes[1, 1].imshow(plt_data_red_sub[4], cmap=cmap, vmin=vmin, vmax=vmax, origin="lower")
axes[1, 1].text(
    47,
    85,
    f"{get_transition_fraction(plt_data_red_sub[4], 0.5) * 100.0:<4.2f}%",
    color="w",
    fontsize=11.0,
    weight="bold",
)

# Add a small grid
for gl in np.linspace(0, 99, 5):
    for ax in axes.flat:
        ax.axhline(gl, c="k", alpha=0.3)
        ax.axvline(gl, c="k", alpha=0.3)

# Remove ticks for all axes
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

fig.supylabel("Weather", fontsize=11)
fig.supxlabel("RL Architecture", fontsize=11)

axes[1][0].set_xlabel("Split-Input")
axes[1][1].set_xlabel("Measured-Input")
axes[0][0].set_ylabel("Cool")
axes[1][0].set_ylabel("Warm")

axes[0, 1].set_xlabel("Building load", fontsize=8)
axes[0, 1].set_ylabel("Aggregate load", fontsize=8)

# Add a horizontal colorbar below the plots
# Add a common colorbar outside the subplots
fig.subplots_adjust(right=0.9)  # Make space for the colorbar
cbar_ax = fig.add_axes([0.93, 0.15, 0.02, 0.7])  # Create colorbar axis
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label("$T^{set,req}_{n,t}$", fontsize=12)
cbar.set_ticks([vmin, vmax])
cbar.set_ticklabels(["-1", "+1"], fontsize=8)

# fig.savefig("export/policy_vis_small.png", dpi=400, bbox_inches="tight")

In [ ]:
import matplotlib as mpl

new_rc_params = {"text.usetex": False, "svg.fonttype": "none"}
mpl.rcParams.update(new_rc_params)
fig.savefig("figures/policy_vis_small.svg", dpi=350, bbox_inches="tight")